# Part 4: Advanced Methods - Gaussian Process Switching Linear Dynamical Systems (gpSLDS)

## Overview

This notebook introduces **gpSLDS**, an extension of LFADS (Part 3) that explicitly models neural dynamics as smooth transitions between multiple local linear regimes.

**Key Innovation:** Use Gaussian Processes to control smooth, continuous switches between K local dynamical modes, rather than hard discrete switches (SLDS) or implicit mode representation (LFADS).

**Time:** 3-4 hours | **Prerequisites:** Parts 0-3

---

## Section 1: Motivation - Why Multiple Modes?

Recall from Part 3 (LFADS): A single linear dynamical system can capture some neural dynamics, but real neural populations often operate in **distinct regimes**:

- Motor planning vs. execution (different dynamical rules)
- Stimulus present vs. absent (different timescales)
- Different brain states (awake, anesthetized, sleep)

**Question:** How can we discover and model these regime switches?

| Model | Dynamics | Mode Switching | Interpretability |
|-------|----------|----------------|------------------|
| LFADS (Part 3) | Single non-linear | Implicit (in RNN) | Low |
| SLDS | Multiple linear | Hard/discrete | High |
| **gpSLDS** | Multiple linear | **Smooth/GP** | **High + Uncertainty** |

**gpSLDS Innovation:** $\pi_k(t)$ = continuous switching probability (not discrete), learned via GP.

---

## Section 2: Gaussian Processes & Kernels

### What is a Gaussian Process?

A GP is a distribution over functions. Any finite set of time points has a multivariate normal distribution.

Key: **Kernel function** $k(t, t')$ controls smoothness.

### RBF Kernel

$$k_{RBF}(t, t') = \sigma^2 \exp\left( -\frac{(t-t')^2}{2\ell^2} \right)$$

- $\ell$: length scale (smoothness)
- Smaller $\ell$: faster changes
- Larger $\ell$: smoother functions


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import pickle

# Import gpSLDS modules
import sys
sys.path.insert(0, '.')
from gpslds_modules import kernels, likelihoods, inference, em, utils

print('OK: All gpSLDS modules loaded')
print(f'JAX version: {jax.__version__}')

---

## Section 3: Core Inference Algorithms

### Kalman Filter + RTS Smoother

E-step of EM: Infer $p(\mathbf{x}_{1:T} | \mathbf{y}_{1:T})$

1. **Forward pass (Kalman filter):** $p(\mathbf{x}_t | \mathbf{y}_{1:t})$
2. **Backward pass (RTS smoother):** $p(\mathbf{x}_t | \mathbf{y}_{1:T})$


In [ ]:
# Create synthetic data
T, D_latent, D_obs = 200, 3, 10
key = jax.random.PRNGKey(42)

# Simple LDS parameters
A = 0.95 * jnp.eye(D_latent)  # Stable dynamics
Q = 0.1 * jnp.eye(D_latent)   # Process noise
C = jax.random.normal(key, (D_obs, D_latent)) * 0.5
R = 0.2 * jnp.eye(D_obs)      # Observation noise

# Generate observations
x_true = jnp.zeros((T, D_latent))
for t in range(1, T):
    key, _ = jax.random.split(key)
    x_true = x_true.at[t].set(A @ x_true[t-1] + jax.random.normal(key, (D_latent,)) * 0.1)

y = C @ x_true.T  # [D_obs, T]
key, _ = jax.random.split(key)
y = (y + jax.random.normal(key, y.shape) * jnp.sqrt(0.2)).T  # [T, D_obs]

print(f'Generated data: y.shape={y.shape}, x_true.shape={x_true.shape}')

In [ ]:
# Run Kalman filter
mu_filt, S_filt, ll_kf = inference.kalman_filter(
    A, Q, C, R, y,
    jnp.zeros(D_latent),
    jnp.eye(D_latent)
)

print(f'Kalman filter log-likelihood: {ll_kf:.4f}')
print(f'Filtered means shape: {mu_filt.shape}')

---

## Section 4: EM Learning

Alternately:
- **E-step:** Infer latent states via Kalman filter + RTS smoother
- **M-step:** Update parameters (C, R, etc.) using smoothed states


In [ ]:
# Fit LDS model via EM
print('Fitting LDS model (simplified gpSLDS with 1 mode)...')
params, x_smooth, ll_hist = em.fit_gpslds(
    y, D_latent=D_latent, num_modes=1,
    max_iter=30, likelihood='gaussian', verbose=True
)

print('\nModel fitting complete!')

In [ ]:
# Visualize results
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# EM convergence
axes[0, 0].plot(ll_hist, 'b-', linewidth=2)
axes[0, 0].set_xlabel('EM Iteration'); axes[0, 0].set_ylabel('Log-Likelihood')
axes[0, 0].set_title('EM Convergence'); axes[0, 0].grid(True, alpha=0.3)

# Latent state recovery
axes[0, 1].plot(x_true[:100, 0], 'k-', label='True', linewidth=2, alpha=0.7)
axes[0, 1].plot(x_smooth[:100, 0], 'r--', label='Inferred', linewidth=2)
axes[0, 1].set_ylabel('Latent State'); axes[0, 1].set_title('Latent State Recovery (dim 0)')
axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

# Observation reconstruction
C_learned = jnp.array(params['C'])
y_recon = (C_learned @ x_smooth.T).T
axes[1, 0].plot(y[:100, 0], 'k-', label='True', alpha=0.7)
axes[1, 0].plot(y_recon[:100, 0], 'r--', label='Reconstructed')
axes[1, 0].set_ylabel('Observation'); axes[1, 0].set_title('Observation Reconstruction (ch 0)')
axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

# Learned C matrix
im = axes[1, 1].imshow(C_learned, aspect='auto', cmap='RdBu_r')
axes[1, 1].set_xlabel('Latent Dim'); axes[1, 1].set_ylabel('Obs Dim')
axes[1, 1].set_title('Learned Observation Matrix C')
plt.colorbar(im, ax=axes[1, 1])

plt.tight_layout(); plt.show()

mse = np.mean((y - np.array(y_recon))**2)
print(f'Reconstruction MSE: {mse:.6f}')

---

## Section 5: Multi-Mode Learning with Demo Data

Now fit with multiple modes using the pre-generated synthetic data:


In [ ]:
# Load pre-generated demo data
try:
    with open('data/demo_data_gaussian.pkl', 'rb') as f:
        data_demo = pickle.load(f)
    y_demo = jnp.array(data_demo['y'])
    x_true_demo = data_demo['x_true']
    mode_seq_demo = data_demo['mode_seq']
    print(f'OK: Loaded demo data')
    print(f'  y shape: {y_demo.shape}')
    print(f'  x_true shape: {x_true_demo.shape}')
    print(f'  num_modes: {data_demo["num_modes"]}')
except FileNotFoundError:
    print('Demo data not found. Generating new data...')
    y_demo, x_true_demo, mode_seq_demo, _ = utils.generate_synthetic_gpslds_data(
        T=500, D_latent=3, D_obs=15, num_modes=3, seed=42
    )
    y_demo = jnp.array(y_demo)

In [ ]:
# Fit multi-mode model
D_latent_demo = x_true_demo.shape[1]
num_modes_demo = len(np.unique(mode_seq_demo))

print(f'Fitting {num_modes_demo}-mode gpSLDS model...')
params_mm, x_smooth_mm, ll_hist_mm = em.fit_gpslds(
    y_demo, D_latent=D_latent_demo, num_modes=num_modes_demo,
    max_iter=50, likelihood='gaussian', verbose=True
)

print('\nMulti-mode model fitting complete!')

In [ ]:
# Visualize multi-mode results
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(ll_hist_mm)
axes[0, 0].set_xlabel('Iteration'); axes[0, 0].set_ylabel('Log-Likelihood')
axes[0, 0].set_title('Multi-Mode EM Convergence'); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(mode_seq_demo[:300], 'o-', markersize=3)
axes[0, 1].set_ylabel('Mode'); axes[0, 1].set_title('True Mode Sequence')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(x_true_demo[:300, 0], 'k-', label='True', alpha=0.7)
axes[1, 0].plot(x_smooth_mm[:300, 0], 'r--', label='Inferred')
axes[1, 0].set_ylabel('Latent (dim 0)'); axes[1, 0].set_title('Latent State Recovery')
axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

C_mm = jnp.array(params_mm['C'])
im = axes[1, 1].imshow(C_mm, aspect='auto', cmap='RdBu_r')
axes[1, 1].set_title('Learned C Matrix'); axes[1, 1].set_xlabel('Latent'); axes[1, 1].set_ylabel('Obs')
plt.colorbar(im, ax=axes[1, 1])

plt.tight_layout(); plt.show()

---

## Summary

### What We Covered

1. **Motivation:** Multiple neural regimes require multiple dynamical modes
2. **Theory:** Gaussian Processes for smooth mode transitions
3. **Algorithms:** Kalman filter & RTS smoother for efficient inference
4. **Learning:** EM algorithm to fit models to data
5. **Practice:** JAX implementation and demonstrations

### Key Takeaway

**gpSLDS** extends LFADS by making mode switches **explicit, smooth, and interpretable**, while maintaining the efficiency of linear dynamics within each mode.

### Next Steps

- Apply to real neural recordings
- Learn all parameters (A, Q, kernel scales) via full EM
- Combine with behavioral data to interpret mode meanings
- Use for online state estimation in closed-loop experiments

### References

- Pandarinath et al. (2018). "Inferring single-trial neural population dynamics using sequential autoencoders." *Nature Methods*.
- Official gpSLDS: https://github.com/lindermanlab/gpslds
- Rasmussen & Williams (2006). *Gaussian Processes for Machine Learning*. MIT Press.
